# Healthcare Member Churn Prediction

End-to-end predictive modeling pipeline:
1. Data exploration & class balance
2. Feature engineering
3. Model training & cross-validation
4. Evaluation (ROC, PR curves, confusion matrix)
5. Feature importance analysis
6. Risk tier assignment for operational use

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

from src.features import prepare_model_data, add_derived_features, NUMERIC_FEATURES
from src.model import (
    cross_validate_models, train_and_evaluate,
    get_feature_importance, assign_risk_tiers
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

## 1. Load & Explore Data

In [ ]:
df = pd.read_csv('../data/member_churn_data.csv')

print(f'Total members: {len(df):,}')
print(f'Churn rate: {df["churned"].mean():.1%}')
print(f'Churned: {df["churned"].sum():,} | Retained: {(~df["churned"].astype(bool)).sum():,}')
print(f'Features: {len(df.columns)}')
df.head(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Class balance
df['churned'].value_counts().plot(kind='bar', ax=axes[0], color=['#27AE60', '#E74C3C'], edgecolor='black')
axes[0].set_title('Churn Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_xticklabels(['Retained', 'Churned'], rotation=0)

# Satisfaction by churn
for label, color in [(0, '#27AE60'), (1, '#E74C3C')]:
    data = df[df['churned'] == label]['satisfaction_score']
    axes[1].hist(data, bins=20, alpha=0.6, color=color, edgecolor='black',
                 label='Churned' if label else 'Retained')
axes[1].set_title('Satisfaction Score by Churn', fontsize=13, fontweight='bold')
axes[1].legend()

# Tenure by churn
for label, color in [(0, '#27AE60'), (1, '#E74C3C')]:
    data = df[df['churned'] == label]['tenure_months']
    axes[2].hist(data, bins=20, alpha=0.6, color=color, edgecolor='black',
                 label='Churned' if label else 'Retained')
axes[2].set_title('Tenure by Churn Status', fontsize=13, fontweight='bold')
axes[2].legend()

plt.tight_layout()
plt.show()

## 2. Feature Engineering

In [ ]:
X, y, feature_names = prepare_model_data(df)

print(f'Feature matrix: {X.shape}')
print(f'Target balance: {y.value_counts().to_dict()}')
print(f'\nFeatures ({len(feature_names)}):')
for i, f in enumerate(feature_names, 1):
    print(f'  {i:2d}. {f}')

In [ ]:
# Top correlations with churn
df_feat = add_derived_features(df)
churn_corr = df_feat[NUMERIC_FEATURES + ['digital_engagement', 'cost_burden', 'service_friction',
                                          'utilization_total', 'engagement_ratio', 'churned']].corr()['churned'].drop('churned').sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#E74C3C' if v > 0 else '#3498DB' for v in churn_corr.values]
ax.barh(churn_corr.index, churn_corr.values, color=colors, edgecolor='black')
ax.set_title('Feature Correlation with Churn', fontsize=14, fontweight='bold')
ax.set_xlabel('Pearson Correlation')
ax.axvline(0, color='gray', linestyle='--')
plt.tight_layout()
plt.show()

## 3. Cross-Validation Comparison

In [ ]:
cv_results = cross_validate_models(X, y)

print('5-Fold Cross-Validation ROC-AUC:\n')
for name, res in cv_results.items():
    print(f'{name:25s}  AUC = {res["mean_auc"]:.3f} ± {res["std_auc"]:.3f}')

fig, ax = plt.subplots(figsize=(10, 5))
names = list(cv_results.keys())
means = [cv_results[n]['mean_auc'] for n in names]
stds = [cv_results[n]['std_auc'] for n in names]
colors = ['#3498DB', '#F39C12', '#27AE60']

ax.bar(names, means, yerr=stds, color=colors, edgecolor='black', capsize=5)
ax.set_title('Model Comparison (5-Fold CV ROC-AUC)', fontsize=13, fontweight='bold')
ax.set_ylabel('ROC-AUC')
ax.set_ylim(0.5, 1.0)
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, m + s + 0.01, f'{m:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Final Model Training & Evaluation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

results = train_and_evaluate(X_train, X_test, y_train, y_test, 'Gradient Boosting')

print(f'Test ROC-AUC: {results["auc"]:.3f}')
print(f'\nClassification Report:')
report_df = pd.DataFrame(results['report']).T
report_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC Curve
fpr, tpr = results['roc_curve']
axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {results["auc"]:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=12)

# Precision-Recall Curve
precision, recall = results['pr_curve']
axes[1].plot(recall, precision, 'r-', linewidth=2)
axes[1].set_title('Precision-Recall Curve', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].axhline(y_test.mean(), color='gray', linestyle='--', alpha=0.5, label='Baseline')
axes[1].legend()

# Confusion Matrix
sns.heatmap(results['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
            xticklabels=['Retained', 'Churned'], yticklabels=['Retained', 'Churned'],
            ax=axes[2])
axes[2].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Actual')
axes[2].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

## 5. Feature Importance

In [ ]:
fi = get_feature_importance(results['model'], feature_names)
top_20 = fi.head(20)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(top_20)), top_20['importance'].values, color='#4C72B0', edgecolor='black')
ax.set_yticks(range(len(top_20)))
ax.set_yticklabels(top_20['feature'].values)
ax.invert_yaxis()
ax.set_title('Top 20 Feature Importances (Gradient Boosting)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance')

plt.tight_layout()
plt.show()

print('\nTop 10 features:')
fi.head(10)

## 6. Risk Tier Assignment

In [ ]:
risk_tiers = assign_risk_tiers(results['y_prob'])
tier_dist = risk_tiers.value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tier_colors = {'Low': '#27AE60', 'Medium': '#F39C12', 'High': '#E67E22', 'Critical': '#E74C3C'}
axes[0].bar(tier_dist.index, tier_dist.values,
            color=[tier_colors[t] for t in tier_dist.index], edgecolor='black')
axes[0].set_title('Risk Tier Distribution (Test Set)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Member Count')
for i, v in enumerate(tier_dist.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Actual churn rate per tier
tier_churn = pd.DataFrame({'tier': risk_tiers, 'churned': y_test.values})
tier_actual = tier_churn.groupby('tier')['churned'].mean().reindex(['Low', 'Medium', 'High', 'Critical'])
axes[1].bar(tier_actual.index, tier_actual.values,
            color=[tier_colors[t] for t in tier_actual.index], edgecolor='black')
axes[1].set_title('Actual Churn Rate by Risk Tier', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Churn Rate')
for i, v in enumerate(tier_actual.values):
    if not np.isnan(v):
        axes[1].text(i, v + 0.01, f'{v:.0%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('Risk tier summary:')
tier_churn.groupby('tier')['churned'].agg(['count', 'sum', 'mean']).rename(
    columns={'count': 'members', 'sum': 'actual_churned', 'mean': 'churn_rate'}
).round(3)

## 7. Key Findings & Recommendations

### Findings
1. **Satisfaction score is the strongest predictor** — members with scores below 5 have 3x churn risk
2. **Service friction (calls + grievances) signals intent to leave** — grievances are an outsized predictor relative to frequency
3. **Digital engagement is protective** — portal/mobile active members churn at lower rates
4. **New members (< 12 months) are vulnerable** — onboarding experience is critical for retention
5. **Gradient Boosting outperforms** other models for this task, balancing precision and recall

### Recommendations
1. **Deploy risk scores to CRM** — flag High/Critical members for proactive outreach
2. **Grievance fast-track program** — escalate resolution for members with any filed grievance
3. **New member onboarding** — dedicated support and check-ins during first 12 months
4. **Digital engagement incentives** — drive portal/app adoption to increase stickiness
5. **Monthly model refresh** — retrain on rolling 12-month data to capture drift